# vLLM 核心架构走读以伪代码和简化实现梳理 vLLM 的核心模块：1. LLMEngine：请求入口和全局协调2. Scheduler：Prefill/Decode 调度 + 抢占3. BlockManager：Paged KV Cache 内存管理4. Worker / ModelRunner：GPU 执行> 腾讯面试必考：vLLM 从请求到响应的完整流程

In [ ]:
import numpy as np
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Set
from enum import Enum
print("Ready.")

## 1) 核心数据结构

In [ ]:
class SequenceStatus(Enum):
    WAITING = "waiting"
    RUNNING = "running"
    SWAPPED = "swapped"
    FINISHED = "finished"

@dataclass
class Sequence:
    seq_id: int
    prompt_tokens: List[int]
    generated_tokens: List[int] = field(default_factory=list)
    status: SequenceStatus = SequenceStatus.WAITING
    logical_blocks: List[int] = field(default_factory=list)
    num_computed_tokens: int = 0

    @property
    def total_len(self):
        return len(self.prompt_tokens) + len(self.generated_tokens)

    @property
    def is_prefill(self):
        return self.num_computed_tokens < len(self.prompt_tokens)

@dataclass
class SchedulerOutput:
    scheduled_prefills: List[Sequence] = field(default_factory=list)
    scheduled_decodes: List[Sequence] = field(default_factory=list)
    preempted: List[Sequence] = field(default_factory=list)
    num_batched_tokens: int = 0

print("Data structures defined: Sequence, SequenceStatus, SchedulerOutput")

## 2) BlockManager：Paged 内存管理

In [ ]:
class SimpleBlockManager:
    """简化版 BlockManager（对应 vLLM 的 BlockSpaceManagerV1）"""

    def __init__(self, num_gpu_blocks: int, num_cpu_blocks: int = 64, block_size: int = 16):
        self.block_size = block_size
        self.gpu_free = list(range(num_gpu_blocks))
        self.cpu_free = list(range(num_cpu_blocks))
        self.block_tables: Dict[int, List[int]] = {}   # seq_id -> [physical_block_id]
        self.ref_counts: Dict[int, int] = {}

    def can_allocate(self, num_tokens: int) -> bool:
        needed = int(np.ceil(num_tokens / self.block_size))
        return len(self.gpu_free) >= needed

    def allocate(self, seq: Sequence) -> List[int]:
        needed = int(np.ceil(seq.total_len / self.block_size))
        blocks = [self.gpu_free.pop() for _ in range(min(needed, len(self.gpu_free)))]
        self.block_tables[seq.seq_id] = blocks
        for b in blocks:
            self.ref_counts[b] = 1
        return blocks

    def can_append(self, seq: Sequence) -> bool:
        current_blocks = len(self.block_tables.get(seq.seq_id, []))
        needed = int(np.ceil(seq.total_len / self.block_size))
        new_blocks = needed - current_blocks
        return new_blocks <= 0 or len(self.gpu_free) >= new_blocks

    def append_slot(self, seq: Sequence):
        current = self.block_tables.get(seq.seq_id, [])
        needed = int(np.ceil(seq.total_len / self.block_size))
        while len(current) < needed and self.gpu_free:
            b = self.gpu_free.pop()
            current.append(b)
            self.ref_counts[b] = 1
        self.block_tables[seq.seq_id] = current

    def free(self, seq: Sequence):
        blocks = self.block_tables.pop(seq.seq_id, [])
        for b in blocks:
            self.ref_counts.pop(b, None)
            self.gpu_free.append(b)

    def swap_out(self, seq: Sequence) -> List[int]:
        gpu_blocks = self.block_tables.pop(seq.seq_id, [])
        cpu_blocks = []
        for gb in gpu_blocks:
            if self.cpu_free:
                cb = self.cpu_free.pop()
                cpu_blocks.append(cb)
            self.ref_counts.pop(gb, None)
            self.gpu_free.append(gb)
        self.block_tables[seq.seq_id] = cpu_blocks  # 记录在 CPU 上的位置
        return cpu_blocks

    @property
    def gpu_free_count(self):
        return len(self.gpu_free)

bm = SimpleBlockManager(num_gpu_blocks=128, block_size=16)
s = Sequence(seq_id=0, prompt_tokens=list(range(100)))
print(f"Before alloc: {bm.gpu_free_count} free blocks")
bm.allocate(s)
print(f"After alloc (100 tokens): {bm.gpu_free_count} free blocks, seq blocks={len(bm.block_tables[0])}")
bm.free(s)
print(f"After free: {bm.gpu_free_count} free blocks")

## 3) Scheduler：调度核心逻辑vLLM 调度策略：**Decode-first**（优先保证已运行请求的进度）

In [ ]:
class SimpleScheduler:
    """简化版 vLLM Scheduler"""

    def __init__(self, block_manager: SimpleBlockManager, max_num_seqs: int = 32,
                 max_num_batched_tokens: int = 2048):
        self.bm = block_manager
        self.max_num_seqs = max_num_seqs
        self.max_num_batched_tokens = max_num_batched_tokens

        self.waiting: List[Sequence] = []
        self.running: List[Sequence] = []
        self.swapped: List[Sequence] = []

    def add_request(self, seq: Sequence):
        seq.status = SequenceStatus.WAITING
        self.waiting.append(seq)

    def schedule(self) -> SchedulerOutput:
        output = SchedulerOutput()

        # Phase 1: Schedule running (decode) sequences
        remaining_running = []
        for seq in self.running:
            if self.bm.can_append(seq):
                self.bm.append_slot(seq)
                output.scheduled_decodes.append(seq)
            else:
                # Preempt: swap out to CPU
                self.bm.swap_out(seq)
                seq.status = SequenceStatus.SWAPPED
                self.swapped.append(seq)
                output.preempted.append(seq)
        self.running = [s for s in output.scheduled_decodes]

        # Phase 2: Schedule new prefills from waiting
        budget_seqs = self.max_num_seqs - len(self.running)
        budget_tokens = self.max_num_batched_tokens - len(self.running)  # decode=1 token each

        new_running = []
        still_waiting = []
        for seq in self.waiting:
            if budget_seqs <= 0 or budget_tokens <= 0:
                still_waiting.append(seq)
                continue
            prompt_len = len(seq.prompt_tokens)
            if prompt_len > budget_tokens:
                still_waiting.append(seq)
                continue
            if not self.bm.can_allocate(prompt_len):
                still_waiting.append(seq)
                continue
            self.bm.allocate(seq)
            seq.status = SequenceStatus.RUNNING
            seq.num_computed_tokens = len(seq.prompt_tokens)
            output.scheduled_prefills.append(seq)
            new_running.append(seq)
            budget_seqs -= 1
            budget_tokens -= prompt_len

        self.waiting = still_waiting
        self.running.extend(new_running)
        output.num_batched_tokens = (
            sum(len(s.prompt_tokens) for s in output.scheduled_prefills)
            + len(output.scheduled_decodes)
        )
        return output

    def finish_request(self, seq_id: int):
        self.running = [s for s in self.running if s.seq_id != seq_id]

# 演示调度过程
bm = SimpleBlockManager(128, block_size=16)
sched = SimpleScheduler(bm, max_num_seqs=8, max_num_batched_tokens=512)

# 提交请求
for i in range(5):
    prompt_len = 50 + i * 20
    sched.add_request(Sequence(seq_id=i, prompt_tokens=list(range(prompt_len))))

# Step 1: 调度
out = sched.schedule()
print(f"Step 1: prefills={len(out.scheduled_prefills)}, decodes={len(out.scheduled_decodes)}, "
      f"preempted={len(out.preempted)}, tokens={out.num_batched_tokens}")
print(f"  Running: {[s.seq_id for s in sched.running]}")
print(f"  Waiting: {[s.seq_id for s in sched.waiting]}")

# 模拟 decode: 每个 running seq 生成 1 token
for s in sched.running:
    s.generated_tokens.append(999)

out2 = sched.schedule()
print(f"\nStep 2: prefills={len(out2.scheduled_prefills)}, decodes={len(out2.scheduled_decodes)}, "
      f"tokens={out2.num_batched_tokens}")

## 4) 完整请求生命周期

In [ ]:
def simulate_vllm_lifecycle(n_requests=10, max_gen_tokens=20, seed=42):
    rng = np.random.default_rng(seed)
    bm = SimpleBlockManager(256, block_size=16)
    sched = SimpleScheduler(bm, max_num_seqs=4, max_num_batched_tokens=256)
    log = []

    # 提交所有请求
    for i in range(n_requests):
        plen = int(rng.integers(30, 100))
        sched.add_request(Sequence(seq_id=i, prompt_tokens=list(range(plen))))

    step = 0
    finished = set()
    while len(finished) < n_requests:
        step += 1
        out = sched.schedule()
        # 模拟 decode
        to_finish = []
        for s in sched.running:
            if not s.is_prefill:
                s.generated_tokens.append(step)
                if len(s.generated_tokens) >= max_gen_tokens:
                    to_finish.append(s.seq_id)

        for sid in to_finish:
            sched.finish_request(sid)
            bm.free(Sequence(seq_id=sid, prompt_tokens=[]))
            bm.block_tables.pop(sid, None)
            finished.add(sid)

        log.append({
            "step": step,
            "prefills": len(out.scheduled_prefills),
            "decodes": len(out.scheduled_decodes),
            "preempted": len(out.preempted),
            "running": len(sched.running),
            "waiting": len(sched.waiting),
            "free_blocks": bm.gpu_free_count,
            "finished": len(finished),
        })

        if step > 200:
            break

    return log

log = simulate_vllm_lifecycle(n_requests=10, max_gen_tokens=15)
print(f"Simulation: {len(log)} steps, final state: {log[-1]}")
print(f"\n{'step':>5} {'prefill':>8} {'decode':>7} {'running':>8} {'waiting':>8} {'free':>6} {'done':>5}")
for l in log[:20]:
    print(f"{l['step']:>5d} {l['prefills']:>8d} {l['decodes']:>7d} {l['running']:>8d} {l['waiting']:>8d} {l['free_blocks']:>6d} {l['finished']:>5d}")

## 5) 面试讲解模板> "vLLM 的核心流程：> 1. **LLMEngine.add_request()** → 请求进入 Scheduler.waiting 队列> 2. **Scheduler.schedule()** → Decode-first 策略：>    - 先保证 running 请求有足够 block 继续 decode>    - 不够 → preempt（swap to CPU / recompute）>    - 再从 waiting 取新请求做 prefill（受 budget 约束）> 3. **BlockManager** → Paged 内存管理：allocate / append_slot / free / swap> 4. **Worker.execute_model()** → GPU 执行 attention + FFN>> 关键设计：Decode-first 避免 starvation，Chunked Prefill 避免大 prompt 独占"### 追问1. **V1 vs V2 BlockManager？** → V2 支持 sliding window, prefix caching2. **Preempt 选谁？** → 最后到达的（LIFO），因为投入最少3. **Chunked Prefill？** → 大 prompt 分多步 prefill，每步不超过 budget4. **和 TRT-LLM 区别？** → vLLM 纯 Python 调度，TRT-LLM 用 C++ runtime + 编译优化